# Risk-aware Capacity Advisor (Reproducible Notebook)

This notebook regenerates capacity-risk artefacts used by the Streamlit dashboard.

**Outputs**:
- `data/processed/capacity_risk_timeseries.csv`
- `reports/figures/capacity_risk_shap_global.csv`
- `reports/figures/capacity_risk_shap_bar.png` (optional visual)


In [ ]:
from pathlib import Path
import os

# Resolve repository root robustly (works in GitHub/VSCode and Colab)
THIS_FILE = Path.cwd()
# If running in Colab after cloning, cwd is repo root; otherwise adjust:
if (THIS_FILE / "app.py").exists():
    REPO_ROOT = THIS_FILE
else:
    # try parent levels
    REPO_ROOT = next((p for p in THIS_FILE.parents if (p / "app.py").exists()), THIS_FILE)

DATA_DIR = REPO_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
INTERIM_DIR = DATA_DIR / "interim"
FIG_DIR = REPO_ROOT / "reports" / "figures"

for d in [PROCESSED_DIR, INTERIM_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("FIG_DIR:", FIG_DIR)

In [ ]:
import numpy as np
import pandas as pd

# Attempt to load an existing capacity time series if present, else build a synthetic demo.
src_candidates = [
    PROCESSED_DIR / "capacity_risk_timeseries.csv",
]
df = None
for p in src_candidates:
    if p.exists():
        df = pd.read_csv(p, parse_dates=["time"])
        print("Loaded existing capacity data:", p, "rows:", len(df))
        break

if df is None or df.empty:
    print("No existing capacity_risk_timeseries.csv found — generating synthetic demo dataset.")
    rng = np.random.default_rng(42)
    idx = pd.date_range("2021-10-25", "2021-11-01", freq="H", tz="UTC")
    beams = ["Beam-A", "Beam-B", "Beam-C"]
    rows = []
    for b in beams:
        base_cap = rng.uniform(200, 260)
        for t in idx:
            demand = base_cap * rng.uniform(0.4, 1.2)
            cap = base_cap * rng.uniform(0.9, 1.1)
            rows.append({"time": t, "beam": b, "capacity": cap, "demand": demand})
    df = pd.DataFrame(rows)

# Ensure types
df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
df = df.dropna(subset=["time"])

In [ ]:
# Compute risk features (used by dashboard)
df["headroom"] = df["capacity"] - df["demand"]
df["risk_index"] = 1.0 - (df["headroom"] / df["capacity"]).clip(0, 1)

# Persist to the location expected by the dashboard
out_ts = PROCESSED_DIR / "capacity_risk_timeseries.csv"
df.sort_values(["beam","time"]).to_csv(out_ts, index=False)
print("Saved:", out_ts, "rows:", len(df))

## Global explainability (SHAP) for capacity risk (demo)
We train a simple surrogate regressor to predict `risk_index` and compute SHAP values.
This yields a global feature importance CSV and a bar plot for the dissertation/dashboard.

In [ ]:
# Train a light surrogate model and compute global SHAP importances
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

feature_cols = ["capacity", "demand", "headroom"]
X = df[feature_cols].values
y = df["risk_index"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

model.fit(X_train, y_train)
r2 = model.score(X_test, y_test)
print("Surrogate R^2:", round(float(r2), 3))

In [ ]:
import numpy as np
import pandas as pd

# SHAP is optional (some environments may not have it)
try:
    import shap
    shap_available = True
except Exception as e:
    shap_available = False
    print("SHAP not available:", e)

out_shap_csv = FIG_DIR / "capacity_risk_shap_global.csv"
out_shap_png = FIG_DIR / "capacity_risk_shap_bar.png"

if shap_available:
    # Use a sample to keep runtime manageable
    n_sample = min(800, len(df))
    X_sample = df[feature_cols].sample(n=n_sample, random_state=42).values

    # Extract fitted RF + transformed data
    scaler = model.named_steps["scaler"]
    rf = model.named_steps["rf"]
    Xs = scaler.transform(X_sample)

    explainer = shap.TreeExplainer(rf)
    shap_vals = explainer.shap_values(Xs)

    # Global importance: mean absolute SHAP per feature
    mean_abs = np.abs(shap_vals).mean(axis=0)
    imp = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap", ascending=False)
    imp.to_csv(out_shap_csv, index=False)
    print("Saved:", out_shap_csv)

    # Bar plot (matplotlib)
    import matplotlib.pyplot as plt
    plt.figure()
    plt.bar(imp["feature"], imp["mean_abs_shap"])
    plt.title("Capacity Risk — Global SHAP importance (surrogate RF)")
    plt.ylabel("Mean |SHAP|")
    plt.tight_layout()
    plt.savefig(out_shap_png, dpi=200)
    plt.show()
    print("Saved:", out_shap_png)
else:
    # Fallback: simple proxy importance based on RF feature_importances_
    rf = model.named_steps["rf"]
    imp = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": rf.feature_importances_}).sort_values("mean_abs_shap", ascending=False)
    imp.to_csv(out_shap_csv, index=False)
    print("Saved fallback importance CSV:", out_shap_csv)

## Done

You can now run the dashboard:

```bash
streamlit run app.py
```